In [2]:
import numpy as np, pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW, CLEAN, SYN = ROOT / "data" / "raw", ROOT / "data" / "clean", ROOT / "data" / "synthetic"
SYN.mkdir(exist_ok=True)

print("Project folder:", ROOT)
for f in ["remitter_clean.csv", "dim_date.csv", "dim_bank.csv"]:
    print(f, "OK" if (CLEAN / f).exists() else "MISSING -> run notebook 01 first")
print("kaggle csv", "OK" if (RAW / "kaggle" / "upi_transactions_2024.csv").exists() else "MISSING")

Project folder: C:\Users\rrron\OneDrive\Desktop\UPI_Failure_Analysis
remitter_clean.csv OK
dim_date.csv OK
dim_bank.csv OK
kaggle csv OK


In [3]:
dim_bank = pd.read_csv(CLEAN / "dim_bank.csv")
fix = {"ONE MOBIKWIK SYSTEMS": "Fintech / non-bank",
       "SARASWAT CO-OPERATIVE BANK": "Co-operative bank",
       "TRI O TECH SOLUTIONS PRIVATE": "Fintech / non-bank"}
dim_bank["bank_type"] = dim_bank.bank_key.map(fix).fillna(dim_bank.bank_type)
dim_bank.to_csv(CLEAN / "dim_bank.csv", index=False)

print(dim_bank.bank_type.value_counts().to_string())
print("\nstill CHECK MANUALLY:", (dim_bank.bank_type == "CHECK MANUALLY").sum())

bank_type
Private sector bank       17
Regional rural bank       13
Public sector bank        12
Small finance bank         9
Payments bank              5
Foreign bank               4
Credit-card UPI entity     3
Fintech / non-bank         2
Co-operative bank          1

still CHECK MANUALLY: 0


In [4]:
rng = np.random.default_rng(42)            # fixed seed -> har run mein same data
N = 400_000

# ---------- 1. REAL inputs: NPCI remitter data -> (month, bank) ----------
rem = pd.read_csv(CLEAN / "remitter_clean.csv")
top15 = rem.groupby("bank_key").volume_mn.sum().nlargest(15).index
rem["bank"] = rem.bank_key.where(rem.bank_key.isin(top15), "OTHER BANKS")
rem["w_bd"], rem["w_td"] = rem.bd_pct * rem.volume_mn, rem.td_pct * rem.volume_mn
bm = rem.groupby(["month", "bank"], as_index=False).agg(vol=("volume_mn", "sum"), w_bd=("w_bd", "sum"), w_td=("w_td", "sum"))
bm["bd_target"] = bm.w_bd / bm.vol / 100
bm["td_target"] = bm.w_td / bm.vol / 100
month_vol = bm.groupby("month").vol.sum()
rows_per_month = (month_vol / month_vol.sum() * N).round().astype(int)
rows_per_month[rows_per_month.idxmax()] += N - rows_per_month.sum()      # total exactly N

# ---------- 2. Realistic MIX from the Kaggle file ----------
k = pd.read_csv(RAW / "kaggle" / "upi_transactions_2024.csv")
k.columns = ["txn_id", "ts", "txn_type", "category", "amount", "status", "sender_age", "receiver_age", "state",
             "sender_bank", "receiver_bank", "device", "network", "fraud", "hour", "dow", "is_weekend"]
def mix(col):
    p = k[col].value_counts(normalize=True); return p.index.to_numpy(), p.to_numpy()
hour_p = k.hour.value_counts(normalize=True).sort_index().to_numpy()
APPS = {"PhonePe": .456, "Google Pay": .323, "Paytm": .080, "Navi": .044, "super.money": .017, "BHIM": .010, "FamApp": .009, "Other apps": .061}
dates = pd.read_csv(CLEAN / "dim_date.csv")
dates["festival_name"] = dates.festival_name.fillna("")

# ---------- 3. OUR ASSUMPTIONS (inhe badal ke dusri beliefs test kar sakte ho) ----------
HOUR_TD = np.ones(24); HOUR_TD[19:22] = 1.6; HOUR_TD[12:14] = 1.2; HOUR_TD[1:5] = 1.3
NETWORK_TD = {"3G": 1.5, "4G": 1.0, "5G": 0.9, "WiFi": 0.9}
AMOUNT_BD = np.array([0.85, 0.90, 1.00, 1.15, 1.50, 2.20])
BINS, LABELS = [0, 100, 500, 1000, 5000, 10000, 1e9], ["<100", "100-499", "500-999", "1k-4.9k", "5k-9.9k", "10k+"]
BD_REASONS = ["Insufficient funds", "Wrong UPI PIN", "Daily limit exceeded", "Invalid / inactive beneficiary", "Declined by bank (risk / other)"]
BD_P = [0.46, 0.24, 0.10, 0.12, 0.08]
TD_REASONS = ["Remitter bank server down / timeout", "NPCI / UPI switch timeout", "PSP / app technical error", "Network connectivity error"]
TD_P = [0.42, 0.18, 0.18, 0.22]
RETRY = {"Insufficient funds": (.30, .40), "Wrong UPI PIN": (.80, .85), "Daily limit exceeded": (.35, .20),
         "Invalid / inactive beneficiary": (.45, .50), "Declined by bank (risk / other)": (.40, .45),
         "Remitter bank server down / timeout": (.70, .75), "NPCI / UPI switch timeout": (.75, .85),
         "PSP / app technical error": (.65, .80), "Network connectivity error": (.70, .85)}   # (P(retry), P(retry works))

# ---------- 4. Rows banao, mahine ke hisaab se ----------
parts = []
for month, n in rows_per_month.items():
    days = dates[dates.month == month].reset_index(drop=True)
    w = (1 + 0.15 * days.intensity) * np.where((days.is_month_end == 1) | (days.day_of_month == 1), 1.10, 1.0)
    di = rng.choice(len(days), n, p=(w / w.sum()).to_numpy())
    b = bm[bm.month == month]
    hour = rng.choice(24, n, p=hour_p)
    v, p = mix("txn_type"); tt = rng.choice(v, n, p=p)
    v, p = mix("network"); nw = rng.choice(v, n, p=p)
    v, p = mix("device"); dv = rng.choice(v, n, p=p)
    v, p = mix("state"); st = rng.choice(v, n, p=p)
    ts = pd.to_datetime(days.date.to_numpy()[di]) + pd.to_timedelta(hour * 3600 + rng.integers(0, 3600, n), unit="s")
    parts.append(pd.DataFrame({
        "timestamp": ts, "date": days.date.to_numpy()[di], "month": month, "hour": hour,
        "day_of_week": days.weekday.to_numpy()[di], "is_month_end": days.is_month_end.to_numpy()[di],
        "is_month_start": days.is_month_start.to_numpy()[di], "intensity": days.intensity.to_numpy()[di],
        "festival_name": days.festival_name.to_numpy()[di],
        "remitter_bank": rng.choice(b.bank.to_numpy(), n, p=(b.vol / b.vol.sum()).to_numpy()),
        "app": rng.choice(list(APPS), n, p=list(APPS.values())), "txn_type": tt,
        "amount": rng.choice(k.amount.to_numpy(), n), "state": st, "device": dv, "network": nw}))
tx = pd.concat(parts, ignore_index=True)
tx["amount_bucket"] = pd.cut(tx.amount, BINS, labels=LABELS, right=False)
tx["is_festival"] = (tx.intensity > 0).astype(int)

# ---------- 5. Failure probability, NPCI se CALIBRATED ----------
tx = tx.merge(bm[["month", "bank", "bd_target", "td_target"]], left_on=["month", "remitter_bank"], right_on=["month", "bank"]).drop(columns="bank")
m_td = pd.Series(HOUR_TD[tx.hour], index=tx.index) * tx.network.map(NETWORK_TD) * (1 + 0.6 * tx.intensity) * np.where(tx.is_month_start == 1, 1.2, 1.0)
m_bd = pd.Series(AMOUNT_BD[tx.amount_bucket.cat.codes] * np.where(tx.is_month_end == 1, 1.25, 1.0) * np.where(tx.hour < 6, 1.15, 1.0), index=tx.index)
grp = [tx.month, tx.remitter_bank]
tx["p_td"] = tx.td_target * m_td / m_td.groupby(grp).transform("mean")     # key trick
tx["p_bd"] = tx.bd_target * m_bd / m_bd.groupby(grp).transform("mean")
u = rng.random(len(tx))
is_td = u < tx.p_td
is_bd = (~is_td) & (u < tx.p_td + tx.p_bd)
tx["status"] = np.where(is_td | is_bd, "Failed", "Success")
tx["failure_type"] = np.select([is_td, is_bd], ["Technical", "Business"], default="")

# ---------- 6. Reason + retry ----------
tx["failure_reason"] = ""
tx.loc[is_bd, "failure_reason"] = rng.choice(BD_REASONS, is_bd.sum(), p=BD_P)
tx.loc[is_td, "failure_reason"] = rng.choice(TD_REASONS, is_td.sum(), p=TD_P)
failed = tx.status == "Failed"
p_retry = tx.failure_reason.map(lambda r: RETRY[r][0] if r else 0)
p_ok = tx.failure_reason.map(lambda r: RETRY[r][1] if r else 0)
retried = failed & (rng.random(len(tx)) < p_retry)
retry_ok = retried & (rng.random(len(tx)) < p_ok)
tx["final_outcome"] = np.select([~failed, retry_ok], ["Completed first time", "Recovered on retry"], default="Lost")

# ---------- 7. Tidy + save ----------
tx = tx.sort_values("timestamp").reset_index(drop=True)
tx.insert(0, "txn_id", ["T" + str(i + 1).zfill(7) for i in range(len(tx))])
tx["timestamp"] = tx.timestamp.dt.strftime("%Y-%m-%d %H:%M:%S")
tx = tx.drop(columns=["p_td", "p_bd", "bd_target", "td_target"])
tx.to_csv(SYN / "upi_synthetic_txns.csv", index=False)                                   # bada file: GitHub par NAHI
tx.sample(20_000, random_state=1).sort_values("txn_id").to_csv(SYN / "upi_synthetic_txns_sample20k.csv", index=False)   # chhota sample: GitHub par JAYEGA

print(len(tx), "rows | failure %.2f%% | technical %.3f%% | business %.2f%%" % (failed.mean() * 100, is_td.mean() * 100, is_bd.mean() * 100))
print(tx.final_outcome.value_counts(normalize=True).round(3).to_dict())
print("columns:", list(tx.columns))

400000 rows | failure 9.67% | technical 0.313% | business 9.36%
{'Completed first time': 0.903, 'Lost': 0.07, 'Recovered on retry': 0.027}
columns: ['txn_id', 'timestamp', 'date', 'month', 'hour', 'day_of_week', 'is_month_end', 'is_month_start', 'intensity', 'festival_name', 'remitter_bank', 'app', 'txn_type', 'amount', 'state', 'device', 'network', 'amount_bucket', 'is_festival', 'status', 'failure_type', 'failure_reason', 'final_outcome']


In [5]:
tx = pd.read_csv(SYN / "upi_synthetic_txns.csv", usecols=["month", "remitter_bank", "status"])
rem = pd.read_csv(CLEAN / "remitter_clean.csv")

real = rem.groupby("month").apply(lambda g: (g.approved_pct * g.volume_mn).sum() / g.volume_mn.sum(), include_groups=False)
syn = tx.groupby("month").apply(lambda g: (g.status == "Success").mean() * 100, include_groups=False)
cmp = pd.DataFrame({"npci": real, "synthetic": syn}).round(2)
cmp["diff"] = (cmp.synthetic - cmp.npci).round(2)
print(cmp.to_string())
print("mean absolute difference: %.2f percentage points" % cmp["diff"].abs().mean())

big = tx[tx.remitter_bank != "OTHER BANKS"].groupby(["month", "remitter_bank"]).apply(
    lambda g: (g.status == "Success").mean() * 100, include_groups=False)
real_b = rem.set_index(["month", "bank_key"]).approved_pct
j = pd.concat([big.rename("synthetic"), real_b.rename("npci")], axis=1, join="inner")
print("bank-month correlation: %.3f  (n=%d)" % (j.synthetic.corr(j.npci), len(j)))

          npci  synthetic  diff
month                          
2025-09  92.23      92.14 -0.09
2025-10  92.06      92.13  0.07
2025-11  91.66      91.78  0.12
2025-12  90.99      90.77 -0.22
2026-01  90.66      90.53 -0.13
2026-02  90.61      90.70  0.09
2026-03  90.19      90.00 -0.19
2026-04  89.91      89.88 -0.03
2026-05  89.55      89.48 -0.07
2026-06  89.40      89.67  0.27
2026-07  88.84      88.85  0.01
2026-08  89.09      89.04 -0.05
mean absolute difference: 0.11 percentage points
bank-month correlation: 0.986  (n=180)
